# 04 · Culminar Fase 3: tres clasificadores comparables

ConvNeXt-Tiny, VMamba-T s1l8 oficial y un híbrido con atención cruzada
bidireccional entre mapas espaciales. No es un ensemble de probabilidades.

**Este notebook entrena los modelos que faltan y genera un cierre verificable.**
La fuente no contiene entrenamientos ejecutados. El cierre solo se produce si
existen tres corridas completas y sus checkpoints coinciden con sus hashes.

v01/v02 quedan como antecedentes exploratorios. Para comparar arquitecturas,
esta tanda parte de pesos ImageNet por rama, con iguales datos, seed, presupuesto,
optimizador y criterio de selección. El híbrido no hereda ConvNeXt ya ajustado
a TBX11K. Así se distingue el efecto de la arquitectura del entrenamiento previo.

Se conservan tres clases (health/sick/TB), 512px y aumentos congelados. Cada
modelo dispone de hasta 12 épocas, warmup 2, paciencia 4 y lote efectivo 32.
El número realmente ejecutado puede diferir por early stopping. La comparación
usa una seed: no prueba superioridad estadística ni constituye evaluación final.
**Test, umbrales, Grad-CAM y selección final corresponden a Fase 5.**

## Montar Drive y recuperar el paquete nuevo

Abra este notebook en Colab con GPU L4/A100/T4 disponible. El código se extrae
en otra carpeta temporal; los notebooks ejecutados 02/03 y sus resultados se
conservan. El nuevo ZIP debe estar en MyDrive/Proyecto_Tuberculosis/.
Guarde las salidas de cada notebook con Ctrl+S.

In [ ]:
import json, sys, hashlib, stat, zipfile, subprocess
from pathlib import Path, PurePosixPath
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BASE = Path('/content/drive/MyDrive/Proyecto_Tuberculosis')
ARCHIVE = DRIVE_BASE / 'tbx11k_fases_3_y_4_colab.zip'
PROJECT_ROOT = Path('/content/tbx11k_complete/project')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as z:
    for item in z.infolist():
        rel = PurePosixPath(item.filename)
        target = (PROJECT_ROOT / item.filename).resolve()
        if rel.is_absolute() or '..' in rel.parts or ':' in item.filename or '\\' in item.filename or stat.S_ISLNK(item.external_attr >> 16) or PROJECT_ROOT.resolve() not in target.parents:
            raise ValueError('Ruta no segura: '+item.filename)
    checksums = json.loads(z.read('phase3_package_checksums.json'))
    for name, expected in checksums.items():
        if hashlib.sha256(z.read(name)).hexdigest() != expected:
            raise ValueError('Archivo alterado en el paquete: '+name)
    z.extractall(PROJECT_ROOT)
# Importa el código nuevo incluso si el runtime había cargado el paquete anterior.
for name in list(sys.modules):
    if name in {'tbxtrain','tbxdata'} or name.startswith(('tbxtrain.','tbxdata.')):
        del sys.modules[name]
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                       '--disable-pip-version-check', '-r', str(PROJECT_ROOT / 'requirements-phase3-suite.txt')])
import torch
if not torch.cuda.is_available(): raise RuntimeError('Seleccione un runtime GPU de Colab.')
from tbxdata.colab import stage_dataset
from tbxdata.validate import validate_manifests
source = DRIVE_BASE / 'TBX11K.zip'
if not source.exists(): source = DRIVE_BASE / 'TBX11K'
DATASET_ROOT = stage_dataset(source, Path('/content/tbx11k_complete/data_cache'), True)
integrity = validate_manifests(PROJECT_ROOT, DATASET_ROOT, all_image_hashes=True)
print('GPU:',torch.cuda.get_device_name(0),'| Integridad:',integrity['status'])
print('Hashes de imágenes verificados:',integrity['all_image_hashes_verified'])

## F3-C1 · Dependencia CUDA y pesos oficiales VMamba

Compila selective scan desde la revisión fijada del repositorio oficial;
conserva el torch/CUDA del runtime y registra el log en /content/tbx_vmamba_build/.
Puede tardar varios minutos. Si falla, revise el log antes de entrenar.
No instale wheels antiguas de Python/torch que no correspondan a su runtime.
Los pesos ImageNet se conservan en Drive/pretrained/. Su carga es estricta.

In [ ]:
# Reemplazo de la celda F3-C1. Ejecutar en el notebook Colab ya preparado.
import json
import subprocess
import sys
from pathlib import Path
import torch

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| CUDA de PyTorch:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
command = [sys.executable, '-u', str(PROJECT_ROOT / 'scripts/setup_vmamba_colab.py'),
           '--project', str(PROJECT_ROOT), '--drive-base', str(DRIVE_BASE)]
log_path = DRIVE_BASE / 'results/vmamba_setup_diagnostic.log'
log_path.parent.mkdir(parents=True, exist_ok=True)
print('Ejecutando instalador con salida en vivo...', flush=True)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
with log_path.open('w', encoding='utf-8') as log:
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
code = process.wait()
print('Código de salida:', code)
print('Registro completo guardado en:', log_path)
if code != 0:
    raise RuntimeError('El instalador falló. Comparte el error mostrado arriba; no continúes con los entrenamientos todavía.')

from tbxtrain.architectures import scan_backend
WEIGHTS = DRIVE_BASE / 'pretrained/vssm1_tiny_0230s_ckpt_epoch_264.pth'
print('Selective scan:', scan_backend())
if scan_backend() == 'torch_reference':
    raise RuntimeError('El instalador terminó, pero el kernel CUDA no está disponible en este proceso.')
CONFIG_SUITE = json.loads((PROJECT_ROOT / 'configs/phase3_suite.json').read_text())
display(CONFIG_SUITE)

## F3-C2 · Smoke de las tres arquitecturas

Un ejemplo por clase/split a 512px verifica forward/backward, actualización y
guardado/recarga. Son corridas técnicas; no forman parte de la tabla de comparación.
FP32 evita depender del escalado AMP que dificultó el smoke anterior.

In [ ]:
from tbxtrain.suite import run_suite_model
RUN_SMOKES = True
if RUN_SMOKES:
    for architecture in CONFIG_SUITE['architectures']:
        smoke = run_suite_model(PROJECT_ROOT, DATASET_ROOT, DRIVE_BASE, architecture,
                                CONFIG_SUITE, weights_path=WEIGHTS, smoke=True)
        print(architecture, smoke['status'])

## F3-C3 · Entrenamientos completos y recuperación automática

Se entrena una arquitectura después de otra. Se publican last.pt e historial
al completar cada época, con RNG y optimizador. Una época parcial se repite.
El registro en results/phase3_suite/ permite reanudar al volver a ejecutar esta
celda con el mismo código/configuración. Las corridas completas se recuperan
sin entrenarlas otra vez. No cambie parámetros a mitad de una tanda controlada.

In [ ]:
RUN_IDS = []
for architecture in CONFIG_SUITE['architectures']:
    marker = DRIVE_BASE / 'results/phase3_suite' / CONFIG_SUITE['protocol_id'] / (architecture+'.json')
    resume = None
    if marker.exists():
        prior = json.loads(marker.read_text())
        run_id = prior['run_id']
        summary_path = DRIVE_BASE / 'experiments' / run_id / 'summary.json'
        if summary_path.exists():
            summary = json.loads(summary_path.read_text())
            if summary['status'] == 'completed_controlled':
                if summary['protocol'] != CONFIG_SUITE: raise RuntimeError('El protocolo cambió. Defina otro protocol_id para una nueva tanda.')
                RUN_IDS.append(run_id)
                print('Recuperada sin reentrenar:',run_id)
                continue
        if (DRIVE_BASE / 'checkpoints' / run_id / 'last.pt').exists(): resume = run_id
    summary = run_suite_model(PROJECT_ROOT, DATASET_ROOT, DRIVE_BASE, architecture,
                               CONFIG_SUITE, weights_path=WEIGHTS, resume_run=resume)
    RUN_IDS.append(summary['run_id'])
print('Corridas disponibles:', RUN_IDS)

## F3-C4 · Comparar y cerrar con evidencia

Requiere las tres arquitecturas, mismo protocolo/datos/código, validación
completa de 1.241 imágenes y checkpoints íntegros. Si algo falta, falla y no
declara la fase terminada. El ranking es preliminar: la decisión final de
Fase 5 incluirá detector, explicabilidad, concordancia y test reservado.

In [ ]:
import pandas as pd
from IPython.display import display, Image
from tbxtrain.suite import close_phase3
from tbxtrain.protocol import atomic_json
CLOSURE, CLOSURE_DIR = close_phase3(DRIVE_BASE, RUN_IDS)
atomic_json(CLOSURE_DIR / 'transfer_integrity.json', integrity)
display(pd.DataFrame(CLOSURE['comparison']).sort_values('macro_f1',ascending=False))
for row in CLOSURE['comparison']:
    print(row['architecture'], row['run_id'])
    folder = DRIVE_BASE / 'experiments' / row['run_id'] / 'figures'
    for name in ['learning_curves.png','confusion_matrix.png']:
        display(Image(filename=str(folder / name)))
print('Fase 3 completa:',CLOSURE['phase3_complete'])
print('Entregar esta carpeta:', CLOSURE_DIR)
print('Guardar este notebook ejecutado con Ctrl+S. Fases 4 y 5 permanecen pendientes.')